# The differential pair, priced out on paper first

Companion to web-app testbench **23 “Differential pair”**: two 10 µm/0.15 µm
SKY130 NFETs, 4 kΩ drain resistors, an ideal tail source stepped over
100/200/400 µA. Before touching the simulator we predict its behaviour two
classic ways, then check both against the BSIM4 truth:

1. **the $g_m/I_D$ method** — look up transconductance efficiency at the
   operating current *density* on the measured device curve from notebook
   01 (`out/fet_extraction.json`). No square-law assumed anywhere.
2. **the square-law story** — $V_{ov}$, $g_m = 2I_D/V_{ov}$, and the
   $\pm\sqrt{2}V_{ov}$ switching range. At $L = 0.15$ µm this *should*
   crack — quantifying where is the point of the exercise.

Run notebook 01 first (or let the fallback cell below re-extract the one
curve it needs).

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from photonflux.nb import Session

s = Session()
EXTRACT = Path("out/fet_extraction.json")
if EXTRACT.exists():
    dev = json.loads(EXTRACT.read_text())["nfet_01v8"]
    print("using notebook 01's extraction")
else:                       # standalone fallback: re-measure the one curve
    b04 = s.load_example("04_sky130_nfet_output_curves")
    tr = s.dcsweep("VG1", "V", 0, 1.8, points=181, schematic=b04)
    idr = np.abs(tr["id_01v8"])
    gm = np.gradient(idr, tr.x)
    k = int(np.argmax(gm))
    dev = {"vgs": tr.x.tolist(), "id": idr.tolist(), "gm": gm.tolist(),
           "vth": float(tr.x[k] - idr[k] / gm[k])}
    print("fet_extraction.json not found — re-extracted nfet_01v8 inline")
vgs_c = np.asarray(dev["vgs"])
id_c = np.asarray(dev["id"])          # A per 1 µm width, Vds = 1.8 V
gm_c = np.asarray(dev["gm"])
VTH = dev["vth"]

## 1. The pen-and-paper table

The pair splits the tail evenly at balance: $I_D = I_T/2$ per side, i.e. a
current *density* of $I_D/10$ µm — that's the x-axis of notebook 01's
$g_m/I_D$ chart. Everything follows:

$$ g_m = \left(\frac{g_m}{I_D}\right)\!\bigg|_{I_D/W} \cdot \frac{I_T}{2},
   \qquad A_{dm} = g_m \,(R_D \,\|\, r_o), \qquad
   V_{ov}^* \equiv \frac{2}{g_m/I_D},\quad
   V_{id}^{max} = \sqrt{2}\,V_{ov}^* $$

Note what we did **not** write: $V_{ov} = V_{GS}-V_{th}$. Check the table —
at these current densities $V_{GS}^*$ sits *below* the extrapolated
$V_{th}$, i.e. the pair operates in **moderate inversion**, where the
square law's overdrive is literally negative and meaningless. The
effective overdrive $V_{ov}^* = 2/(g_m/I_D)$ (an identity for a true
square-law device) is the honest stand-in, and it comes straight off the
measured curve. ($r_o$ enters as a ~10 % correction; $\lambda \approx
0.3$/V from notebook 01's output-curve fit. Second caveat: the lookup was
measured at $V_{DS}$ = 1.8 V, the pair sits lower — a known ~10 %
systematic of the quick method.)

In [ ]:
W_PAIR, RD, VDD, VCM = 10.0, 4e3, 1.8, 0.9
LAM = 0.3
tails = [100e-6, 200e-6, 400e-6]
hand = {}
print(f"{'IT':>6s} {'Id/W':>10s} {'Vgs*':>7s} {'Vgs*-Vth':>8s} {'gm/Id':>7s} "
      f"{'Vov*':>6s} {'gm':>8s} {'Adm':>6s} {'√2·Vov*':>8s}")
for it in tails:
    id_side = it / 2
    id_1um = id_side / W_PAIR                 # target on the W=1 µm curve
    vgs_star = float(np.interp(id_1um, id_c, vgs_c))
    eff = float(np.interp(vgs_star, vgs_c, gm_c / np.maximum(id_c, 1e-12)))
    gm = eff * id_side
    ro = 1 / (LAM * id_side)
    adm = gm * (RD * ro / (RD + ro))
    vov = 2 / eff                             # effective square-law overdrive
    hand[it] = {"vgs": vgs_star, "vov": vov, "gm": gm, "adm": adm,
                "vid_max": np.sqrt(2) * vov}
    print(f"{it * 1e6:4.0f}µA {id_1um * 1e6:7.1f}µA/µm {vgs_star:7.3f} "
          f"{vgs_star - VTH:8.3f} {eff:5.1f}/V {vov * 1e3:4.0f}mV "
          f"{gm * 1e3:6.2f}mS {adm:6.1f} {np.sqrt(2) * vov * 1e3:6.0f}mV")

## 2. Now ask the simulator

The stored analysis sweeps `VIN` 0.5 → 1.3 V with the tail stepped —
exactly the browser's Run button. `voutp/voutn` arrive as one trace per
tail value; the differential output and its slope at balance give the
measured gain.

In [ ]:
bench = s.load_example("23_diff_pair_transfer")
res = s.run(schematic=bench)
vin = res.x
vid = vin - VCM

fam_p = res.family("voutp")
fam_n = res.family("voutn")
meas = {}
for (np_, yp), yn in zip(fam_p.items(), fam_n.values()):
    it = float(np_.split("@")[1].strip().rstrip("V"))   # step label = IT
    vod = yp - yn
    gain = np.abs(np.gradient(vod, vid))
    meas[it] = {"vod": vod, "gain": float(gain[np.abs(vid).argmin()]),
                "cm": float(yp[np.abs(vid).argmin()])}

print(f"{'IT':>6s} {'Adm hand':>9s} {'Adm sim':>8s} {'err':>6s} "
      f"{'Vout,cm sim':>11s} {'VDD-RD·IT/2':>11s}")
for it in tails:
    h, m = hand[it], meas[it]
    err = m["gain"] / h["adm"] - 1
    cm_pred = VDD - RD * it / 2
    print(f"{it * 1e6:4.0f}µA {h['adm']:9.1f} {m['gain']:8.1f} "
          f"{err * 100:5.1f}% {m['cm']:11.3f} {cm_pred:11.3f}")
    assert abs(err) < 0.15, (it, err)
    assert abs(m["cm"] - cm_pred) < 0.02

The $g_m/I_D$ prediction lands within a few percent across a 4× tail-current
range — the whole point of the method: it inherits the real device physics
(moderate inversion, short-channel $g_m$ degradation) through the measured
curve, no model math required.

## 3. The full transfer, and where the square law cracks

Square-law theory says the pair steers current as

$$ \Delta I(V_{id}) = \frac{\beta}{2} V_{id}
   \sqrt{\frac{4 I_T}{\beta} - V_{id}^2}, \qquad
   \beta = \frac{I_T}{V_{ov}^{*2}},
   \qquad |V_{id}| \le \sqrt{2}\,V_{ov}^* $$

then clips at $\pm I_T$ — with $\beta$ chosen so the square law reproduces
the *measured* $g_m$ at balance (the $V_{ov}^*$ trick above; the most
charitable square law there is). Overlaying it (dashed) on the BSIM4
transfer: the shape and the plateaus are right, but the real device's
soft moderate-inversion turn-on stretches the transition well past the
square-law clip point.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
for it, c in zip(tails, ("C0", "C1", "C2")):
    ax.plot(vid * 1e3, meas[it]["vod"], color=c,
            label=f"BSIM4, IT = {it * 1e6:.0f} µA")
    beta = it / hand[it]["vov"] ** 2
    vlim = np.sqrt(2) * hand[it]["vov"]
    di = np.where(np.abs(vid) < vlim,
                  beta / 2 * vid * np.sqrt(np.maximum(4 * it / beta
                                                      - vid ** 2, 0)),
                  np.sign(vid) * it)
    ax.plot(vid * 1e3, -RD * di, "--", color=c, lw=1,
            label=f"square law (√2·Vov* = ±{vlim * 1e3:.0f} mV)")
    plateau = np.abs(meas[it]["vod"]).max()
    assert abs(plateau / (RD * it) - 1) < 0.05, (it, plateau)
ax.set_xlabel("V_id [mV]"), ax.set_ylabel("V_od = voutp − voutn [V]")
ax.legend(fontsize=7), ax.grid(alpha=.3)
ax.set_title("fully switched plateaus hit ±R_D·I_T exactly; "
             "the knees are softer than square law")
fig.tight_layout()

In [ ]:
# quantify the crack: input range to steer 80% of the tail
for it in tails:
    vod = meas[it]["vod"]
    # first Vid where |Vod| crosses 80% of the full plateau
    idx = np.where(np.abs(vod) >= 0.8 * RD * it)[0]
    v80_sim = float(np.abs(vid[idx]).min()) if len(idx) else float("nan")
    v80_sq = np.sqrt(2) * hand[it]["vov"] * np.sqrt(1 - np.sqrt(1 - 0.8 ** 2))
    print(f"IT = {it * 1e6:3.0f} µA: 80%-steer at |Vid| = {v80_sim * 1e3:.0f} mV "
          f"(square law {v80_sq * 1e3:.0f} mV)")

---
**Takeaways.** Two numbers a designer needs — gain and steering range —
and two grades of theory: the $g_m/I_D$ lookup nails the gain to a few
percent because it *is* the device data; the square law gets the shape and
the plateaus right but misjudges the transition width, because a 0.15 µm
channel at these densities lives in moderate inversion where
$I_D \propto (V_{GS}-V_{th})^2$ simply isn't the law.

**Things to try**

* Re-size the pair from the $g_m/I_D$ chart: pick a density for
  $g_m/I_D = 15$/V, set `MN1/MN2.w_um` accordingly in the browser, and
  re-run this notebook against the live canvas (`s.run()` with no
  arguments uses your edits).
* Add 100 Ω source-degeneration resistors and watch the gain trade
  linearly for input range.
* Step `VCM` instead of `IT` — with an ideal tail the transfer barely
  moves; replace the tail with a real current mirror (testbench
  `42_nmos_current_mirror`) and it won't.